# 19 | H-JEPA: Plan from high level to low level

## Learning goal
Trace how a high-level goal can become a lower-level subgoal and then a short action sequence.

## Paper context
H-JEPA trains action-conditioned JEPA models at several levels. Each level predicts farther ahead in its own learned latent space. Planning runs from the top level down: a prediction at one level becomes a subgoal for the level below. See the [H-JEPA v1 paper](https://arxiv.org/abs/2610.06805v1) and [pinned official code](https://github.com/kevinghst/H-JEPA/tree/97054ecaff83e13839195ec4b92d74c08a0127c5).

## What this notebook does
It uses hand-written state summaries and a hand-written subgoal chain to explain the idea. The example is an **educational analogy**. It does not implement a JEPA, train a hierarchy, or reproduce a paper result. It uses only Python's standard library and runs on CPU.

**Prerequisites:** basic Python and the idea of a state and an action.
**Expected output:** a small example of level-specific summaries and a top-down subgoal chain.


## 1. A toy example with fast and slow details

Imagine a corridor. The room and route position change slowly. A small visual flicker changes quickly. In a learned hierarchy, a higher level may keep the slower task detail while dropping fast detail that does not help its longer prediction. Here, the level summaries below are defined by hand to make that idea visible.


In [1]:
observations = [
    {"room": "A", "position": 2, "visual_flicker": 0},
    {"room": "A", "position": 2, "visual_flicker": 1},
]

# Hand-authored summaries stand in for different latent spaces.
def level_1(obs):
    return (obs["room"], obs["position"], obs["visual_flicker"])

def level_2(obs):
    return (obs["room"], obs["position"] // 2)

def level_3(obs):
    return (obs["room"],)

views = {
    "level 1": [level_1(obs) for obs in observations],
    "level 2": [level_2(obs) for obs in observations],
    "level 3": [level_3(obs) for obs in observations],
}
for name, values in views.items():
    print(f"{name}: {values}")

assert views["level 1"][0] != views["level 1"][1]
assert views["level 2"][0] == views["level 2"][1]
assert views["level 3"][0] == views["level 3"][1]


level 1: [('A', 2, 0), ('A', 2, 1)]
level 2: [('A', 1), ('A', 1)]
level 3: [('A',), ('A',)]


The first level keeps the flicker. The higher toy summaries drop it. **This is a property of the rules written above.** It is not evidence that a trained H-JEPA hierarchy learns this representation. The paper studies this behavior in learned latent spaces; this notebook only gives it a small, inspectable analogy.


## 2. Pass a goal down the hierarchy

The next table is also hand-written. The horizon values show only the relative idea that higher levels plan farther ahead. They are not H-JEPA configuration values.


In [2]:
plan = [
    {"level": 3, "toy_horizon": 8, "goal": "goal region", "subgoal": "hall region"},
    {"level": 2, "toy_horizon": 4, "goal": "hall region", "subgoal": "doorway"},
    {"level": 1, "toy_horizon": 1, "goal": "doorway", "subgoal": ["move", "turn", "move"]},
]

for step in plan:
    print(
        f"L{step['level']} (toy horizon {step['toy_horizon']}): "
        f"goal={step['goal']!r} -> next={step['subgoal']!r}"
    )

assert plan[0]["subgoal"] == plan[1]["goal"]
assert plan[1]["subgoal"] == plan[2]["goal"]
assert plan[-1]["subgoal"] == ["move", "turn", "move"]


L3 (toy horizon 8): goal='goal region' -> next='hall region'
L2 (toy horizon 4): goal='hall region' -> next='doorway'
L1 (toy horizon 1): goal='doorway' -> next=['move', 'turn', 'move']


The example follows the planning direction: the highest level picks an abstract region; the next level receives that target and picks a nearer waypoint; the lowest level turns its target into primitive actions. A real H-JEPA uses learned encoders and predictors at each level and plans with its configured latent costs. This toy uses fixed strings and no learned model.

## 3. Paper result and evidence boundary

The H-JEPA v1 paper reports 73.3% ± 3.5 success for a three-level planner and 18.0% ± 3.5 for flat LeWM on Visual AntMaze, across three model seeds. These are **paper-reported values**, not outputs from this notebook. The local [preflight report](../docs/reproductions/h-jepa-antmaze-preflight.md) records that the official evaluator stopped at renderer startup before producing a task result.

This notebook does not use AntMaze, checkpoints, or the official planner. It does not compare planning compute or validate the reported success rates. A local success chain in this toy would show only that the hand-written example runs.
